# OmniCare Clinical Copilot - Notebook 4: Discharge Summary (Multi-Agent)

**Pipeline:** ClinicalOrchestrator -> DischargeAgent -> Aggregate All Stages -> Discharge Summary + ICD-10 Codes

This notebook uses the multi-agent architecture to generate a comprehensive discharge summary:
1. Load complete encounter data from **Firestore** (all prior stages)
2. Review aggregated clinical data including **HeAR respiratory findings**
3. Run the **DischargeAgent** via the orchestrator
4. Extract and validate **ICD-10 codes** (MCP tool lookup + built-in fallback)
5. View the full **agent activity log** from Firestore
6. Export final documents (**Markdown + JSON**)
7. Mark encounter as **discharged** in Firestore

**Prerequisites:**
- Notebook 00 must be run first (provides `orchestrator` and `models`)
- At least Notebooks 01-02 should have been run for the same encounter
- Firestore must be connected (configured in Notebook 00)

## 0. Colab Bootstrap (run this first)

Auto-detects environment. In Colab, clones the private repo using your GitHub PAT.

**One-time setup:** In Colab, go to the **Key icon** in the left sidebar > Add a secret named `GITHUB_PAT` with your [GitHub Personal Access Token](https://github.com/settings/tokens).

In [ ]:
# ===========================================================
# Colab Bootstrap - run this cell first (works locally too)
# ===========================================================
import os, sys

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_DIR = '/content/omnicare-clinical-copilot'

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        token = userdata.get('GITHUB_PAT')
        repo_url = f'https://{token}@github.com/Yashground/omnicare-clinical-copilot.git'
        os.system(f'git clone {repo_url} {REPO_DIR}')
    NOTEBOOKS_DIR = os.path.join(REPO_DIR, 'notebooks')
    os.makedirs('/content/encounters', exist_ok=True)
    os.makedirs('/content/sample_data', exist_ok=True)
else:
    NOTEBOOKS_DIR = os.path.dirname(os.path.abspath('__file__'))

if NOTEBOOKS_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOKS_DIR)

print(f'Environment ready | Colab: {IN_COLAB} | Notebooks dir: {NOTEBOOKS_DIR}')

## 1. Load Complete Encounter from Firestore

In [ ]:
import json
from datetime import datetime

from utils.firestore_db import (
    load_encounter, list_encounters, update_encounter_status,
    get_agent_logs, get_vitals, get_medications, get_conditions,
)
from utils.fhir_helpers import format_vitals_summary

# --- Verify orchestrator is available from Notebook 00 ---
try:
    _ = orchestrator
    _ = models
    print("Orchestrator and models loaded from Notebook 00.")
except NameError:
    raise RuntimeError(
        "orchestrator / models not found. "
        "Please run Notebook 00 (Setup & Models) first."
    )

# --- List available encounters and select the most recent ---
available = list_encounters()
print(f"Encounters in Firestore: {len(available)}")
for eid in available:
    print(f"  - {eid}")

encounter_id = available[-1] if available else None
if encounter_id is None:
    raise RuntimeError("No encounters found in Firestore. Run Notebooks 01-03 first.")

# --- Load full encounter document ---
encounter = load_encounter(encounter_id)
print(f"\nLoaded encounter: {encounter_id}")
print(f"Patient: {encounter['patient']['name']}")
print(f"MRN:     {encounter['patient']['mrn']}")
print(f"DOB:     {encounter['patient']['dob']}")
print(f"Status:  {encounter.get('status', 'unknown')}")

# --- Show stage completion ---
print("\nStage completion:")
for stage_name, stage_data in encounter["stages"].items():
    has_data = stage_data.get("timestamp") is not None
    status = "COMPLETE" if has_data else "PENDING"
    print(f"  {stage_name:15s} [{status}]")

## 2. Review Aggregated Clinical Data

Display all clinical data collected across the encounter stages, including HeAR respiratory findings.

In [ ]:
stages = encounter["stages"]
consultation = stages.get("consultation", {})
admission    = stages.get("admission", {})
radiology    = stages.get("radiology", {})

# ---- SOAP Note ----
soap = consultation.get("soap_note", {})
soap_text = ""
if isinstance(soap, dict):
    for section in ["subjective", "objective", "assessment", "plan"]:
        content = soap.get(section, "")
        if content:
            soap_text += f"**{section.title()}:** {content}\n\n"
else:
    soap_text = str(soap)
if not soap_text.strip():
    soap_text = "No consultation note available."

# ---- Admission Note ----
admission_note = admission.get("admission_note", "No admission note available.")

# ---- Vitals (from Firestore subcollection) ----
vitals_history = admission.get("vitals_history", [])
vitals_trend = format_vitals_summary(vitals_history)

# Also pull latest vitals from Firestore subcollection
try:
    fs_vitals = get_vitals(encounter_id)
    print(f"Firestore vitals subcollection: {len(fs_vitals)} readings")
except Exception:
    fs_vitals = []

# ---- Anomalies ----
anomalies = admission.get("anomalies", [])

# ---- Radiology Reports ----
radiology_reports = ""
for i, report in enumerate(radiology.get("reports", []), 1):
    findings = report.get("findings", "No findings")
    radiology_reports += f"\nReport {i}:\n{findings}\n"
if not radiology_reports.strip():
    radiology_reports = "No radiology reports available."

# ---- HeAR Respiratory Findings ----
hear_findings = consultation.get("hear_findings", [])

# ---- Display ----
print("=" * 70)
print("AGGREGATED CLINICAL DATA")
print("=" * 70)

print(f"\n{'---'*20}")
print(f"CONSULTATION  (timestamp: {consultation.get('timestamp', 'N/A')})")
print(f"{'---'*20}")
print(f"Transcript length: {len(consultation.get('transcript', '') or '')} chars")
print(soap_text[:500] + ("..." if len(soap_text) > 500 else ""))

if hear_findings:
    print(f"\n  HeAR Respiratory Events: {len(hear_findings)}")
    for evt in hear_findings:
        print(f"    - {evt.get('event_type','?')} at "
              f"{evt.get('start_sec',0):.1f}-{evt.get('end_sec',0):.1f}s "
              f"(confidence: {evt.get('confidence',0):.0%})")
else:
    print("\n  HeAR Respiratory Events: None recorded")

print(f"\n{'---'*20}")
print(f"ADMISSION  (timestamp: {admission.get('timestamp', 'N/A')})")
print(f"{'---'*20}")
print(f"Vitals: {len(vitals_history)} readings")
print(vitals_trend[:400] if vitals_trend else "  No vitals data.")
if anomalies:
    print(f"\n  Vital Sign Anomalies: {len(anomalies)}")
    for a in anomalies:
        print(f"    [{a['status']}] {a['message']}")
print(f"\nConditions: {len(admission.get('conditions', []))}")
for c in admission.get("conditions", []):
    print(f"  - {c.get('display', 'N/A')}")
print(f"Medications: {len(admission.get('medications', []))}")
for m in admission.get("medications", []):
    print(f"  - {m.get('display', 'N/A')}")
print(f"\nAdmission Note: {len(str(admission_note))} chars")
print(str(admission_note)[:400] + ("..." if len(str(admission_note)) > 400 else ""))

print(f"\n{'---'*20}")
print(f"RADIOLOGY  (timestamp: {radiology.get('timestamp', 'N/A')})")
print(f"{'---'*20}")
print(f"Images: {len(radiology.get('images', []))}")
print(f"Reports: {len(radiology.get('reports', []))}")
print(radiology_reports[:400] + ("..." if len(radiology_reports) > 400 else ""))

## 3. Run Discharge Agent

The `DischargeAgent` (via `orchestrator.run_discharge`) performs:
1. Loads the full encounter from Firestore
2. Aggregates SOAP note, admission note, vitals, radiology, and HeAR findings
3. Calls MedGemma to generate a comprehensive discharge summary
4. Extracts ICD-10 codes from diagnoses using built-in mapping
5. Persists everything back to Firestore
6. Marks the encounter status as "discharged"

In [ ]:
# Point the orchestrator at this encounter
orchestrator.encounter_id = encounter_id

print(f"Running DischargeAgent for encounter: {encounter_id}")
print("This aggregates all stages and generates the discharge summary with MedGemma.\n")

# Run the discharge phase via the orchestrator
discharge_result = orchestrator.run_discharge()

# Store for later sections
discharge_summary = discharge_result["discharge_summary"]
icd10_codes       = discharge_result["icd10_codes"]
meds_at_discharge = discharge_result.get("medications_at_discharge", [])

## 4. Display Discharge Summary

In [ ]:
print("=" * 70)
print("DISCHARGE SUMMARY")
print("=" * 70)
print(f"Encounter: {encounter_id}")
print(f"Patient:   {encounter['patient']['name']}")
print(f"Generated: {datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}")
print("-" * 70)
print(discharge_summary)
print("-" * 70)

# Medications at discharge
if meds_at_discharge:
    print(f"\nMedications at Discharge ({len(meds_at_discharge)}):")
    for med in meds_at_discharge:
        print(f"  - {med}")

# Follow-up instructions (set by DischargeAgent)
encounter_updated = load_encounter(encounter_id)
follow_up = encounter_updated["stages"]["discharge"].get("follow_up", "N/A")
print(f"\nFollow-up: {follow_up}")
print(f"Encounter status: {encounter_updated.get('status', 'unknown')}")

## 5. ICD-10 Codes

Show the ICD-10 codes extracted by the DischargeAgent's built-in mapping.
Then optionally validate or look up additional codes using the ICD-10 MCP tools.

In [ ]:
print("=" * 70)
print("ICD-10 CODE ASSIGNMENTS")
print("=" * 70)

# --- Codes from DischargeAgent built-in mapping ---
print(f"\nCodes extracted by DischargeAgent ({len(icd10_codes)}):\n")
for code_info in icd10_codes:
    print(f"  {code_info['code']:10s}  {code_info['description']}")

if not icd10_codes:
    print("  No ICD-10 codes auto-matched. Manual review needed.")

# --- Diagnoses source text ---
conditions = admission.get("conditions", [])
assessment = soap.get("assessment", "") if isinstance(soap, dict) else ""

if conditions:
    print(f"\nSource conditions ({len(conditions)}):")
    for c in conditions:
        print(f"  - {c.get('display', 'N/A')}")

if assessment:
    print(f"\nSOAP Assessment (used for code matching):")
    print(f"  {assessment[:400]}")

# --- Optional: Validate codes using ICD-10 MCP tool ---
# Uncomment the block below if the ICD-10 MCP server is available
# in your Claude environment. This calls the external ICD-10 lookup tool.
#
# print("\n--- ICD-10 MCP Validation ---")
# for code_info in icd10_codes:
#     # Call: mcp__claude_ai_ICD-10_Codes__validate_code(code=code_info["code"])
#     # Call: mcp__claude_ai_ICD-10_Codes__lookup_code(code=code_info["code"])
#     pass

print(f"\nNote: Use ICD-10 MCP tools (search_codes, validate_code, lookup_code)")
print("for production-grade code validation and additional code discovery.")

## 6. Agent Activity Log

Retrieve the complete audit trail of agent actions for this encounter from Firestore.
This shows every step taken by each agent throughout the patient journey.

In [ ]:
# Retrieve agent logs from Firestore via the orchestrator
agent_logs = orchestrator.get_agent_logs()

# Fallback: query Firestore directly if orchestrator method returns empty
if not agent_logs:
    try:
        agent_logs = get_agent_logs(encounter_id)
    except Exception as e:
        print(f"Could not retrieve agent logs: {e}")
        agent_logs = []

print("=" * 70)
print(f"AGENT ACTIVITY LOG  ({len(agent_logs)} entries)")
print("=" * 70)

if agent_logs:
    # Group by agent for readability
    from collections import defaultdict
    by_agent = defaultdict(list)
    for log in agent_logs:
        by_agent[log.get("agent", "unknown")].append(log)

    for agent_name, logs in by_agent.items():
        print(f"\n  [{agent_name}] ({len(logs)} actions)")
        for log in logs:
            ts = log.get("timestamp", "?")
            action = log.get("action", "?")
            details = log.get("details", "")
            detail_str = f" -- {details}" if details else ""
            print(f"    {ts}  {action}{detail_str}")
else:
    print("\n  No agent logs found for this encounter.")
    print("  Agent logs are written to Firestore by each agent's _log() method.")

## 7. Export Final Documents

Export both a formatted **Markdown** discharge report and the full encounter as **JSON**.

In [ ]:
# Reload the final encounter state from Firestore (includes discharge stage)
final_encounter = load_encounter(encounter_id)
final_stages = final_encounter["stages"]
patient = final_encounter["patient"]

# --- Determine output directory ---
import os
if IN_COLAB:
    output_dir = "/content/encounters"
else:
    output_dir = os.path.join(NOTEBOOKS_DIR, "..", "output", "encounters")
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# Export 1: Markdown Discharge Report
# ============================================================
md_path = os.path.join(output_dir, f"{encounter_id}_discharge_report.md")

with open(md_path, "w", encoding="utf-8") as f:
    f.write(f"# Discharge Report: {encounter_id}\n\n")
    f.write(f"**Patient:** {patient['name']}\n")
    f.write(f"**MRN:** {patient['mrn']}\n")
    f.write(f"**DOB:** {patient['dob']}\n")
    f.write(f"**Status:** {final_encounter.get('status', 'discharged')}\n")
    f.write(f"**Generated:** {datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}\n\n")
    f.write("---\n\n")

    # Consultation SOAP
    f.write("## Consultation SOAP Note\n\n")
    soap_final = final_stages["consultation"].get("soap_note", {})
    if isinstance(soap_final, dict):
        for k, v in soap_final.items():
            if v:
                f.write(f"**{k.title()}:** {v}\n\n")
    else:
        f.write(str(soap_final) + "\n\n")

    # HeAR Findings
    hear = final_stages["consultation"].get("hear_findings", [])
    if hear:
        f.write("### HeAR Respiratory Findings\n\n")
        for evt in hear:
            f.write(f"- {evt.get('event_type','?')} at "
                    f"{evt.get('start_sec',0):.1f}-{evt.get('end_sec',0):.1f}s "
                    f"(confidence: {evt.get('confidence',0):.0%})\n")
        f.write("\n")

    f.write("---\n\n")

    # Admission Note
    f.write("## Admission Note\n\n")
    f.write(str(final_stages["admission"].get("admission_note", "N/A")) + "\n\n")

    # Vitals Anomalies
    anom = final_stages["admission"].get("anomalies", [])
    if anom:
        f.write("### Vital Sign Anomalies\n\n")
        for a in anom:
            f.write(f"- **[{a['status']}]** {a['message']}\n")
        f.write("\n")

    f.write("---\n\n")

    # Radiology Reports
    f.write("## Radiology Reports\n\n")
    for i, r in enumerate(final_stages["radiology"].get("reports", []), 1):
        f.write(f"### Report {i}\n\n")
        f.write(str(r.get("findings", "N/A")) + "\n\n")

    f.write("---\n\n")

    # Discharge Summary
    f.write("## Discharge Summary\n\n")
    f.write(str(final_stages["discharge"].get("summary", "N/A")) + "\n\n")

    f.write("---\n\n")

    # ICD-10 Codes
    f.write("## ICD-10 Codes\n\n")
    for code in final_stages["discharge"].get("icd10_codes", []):
        f.write(f"- **{code['code']}**: {code['description']}\n")
    f.write("\n")

    # Medications at Discharge
    meds = final_stages["discharge"].get("medications_at_discharge", [])
    if meds:
        f.write("## Medications at Discharge\n\n")
        for med in meds:
            f.write(f"- {med}\n")
        f.write("\n")

    # Follow-up
    f.write("## Follow-up\n\n")
    f.write(str(final_stages["discharge"].get("follow_up", "N/A")) + "\n")

print(f"Markdown report exported: {md_path}")

# ============================================================
# Export 2: Full Encounter JSON
# ============================================================
json_path = os.path.join(output_dir, f"{encounter_id}.json")

# Include agent logs in the JSON export
export_data = dict(final_encounter)
export_data["agent_logs"] = agent_logs
export_data["exported_at"] = datetime.utcnow().isoformat()

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(export_data, f, indent=2, default=str)

print(f"JSON encounter exported:  {json_path}")
print(f"\nBoth files ready for download from the file browser.")

## 8. View Complete Encounter Record

Final view of the full encounter from Firestore, showing all four stages and their data.

In [ ]:
# Final reload from Firestore to show the complete record
final = load_encounter(encounter_id)

print("=" * 70)
print(f"COMPLETE ENCOUNTER RECORD: {encounter_id}")
print(f"Patient: {final['patient']['name']}  |  MRN: {final['patient']['mrn']}  |  DOB: {final['patient']['dob']}")
print(f"Status:  {final.get('status', 'unknown')}  |  Created: {final.get('created_at', 'N/A')}")
print("=" * 70)

s = final["stages"]

# --- Stage 1: Consultation ---
print(f"\n{'-'*50}")
print("STAGE 1: CONSULTATION")
print(f"{'-'*50}")
print(f"Timestamp: {s['consultation'].get('timestamp', 'N/A')}")
transcript = s['consultation'].get('transcript', '')
print(f"Transcript: {len(transcript or '')} chars")
if transcript:
    print(f"  {str(transcript)[:250]}...")
soap_d = s['consultation'].get('soap_note', {})
if isinstance(soap_d, dict):
    for k, v in soap_d.items():
        if v:
            print(f"  {k.upper()}: {str(v)[:200]}{'...' if len(str(v)) > 200 else ''}")
hear = s['consultation'].get('hear_findings', [])
if hear:
    print(f"  HeAR events: {len(hear)}")
    for e in hear:
        print(f"    - {e.get('event_type','?')} ({e.get('confidence',0):.0%})")

# --- Stage 2: Admission ---
print(f"\n{'-'*50}")
print("STAGE 2: ADMISSION")
print(f"{'-'*50}")
print(f"Timestamp:  {s['admission'].get('timestamp', 'N/A')}")
print(f"Vitals:     {len(s['admission'].get('vitals_history', []))} readings")
print(f"Conditions: {len(s['admission'].get('conditions', []))}")
print(f"Medications:{len(s['admission'].get('medications', []))}")
print(f"Anomalies:  {len(s['admission'].get('anomalies', []))}")
adm_note = s['admission'].get('admission_note', 'N/A')
print(f"Admission Note: {str(adm_note)[:250]}{'...' if len(str(adm_note)) > 250 else ''}")

# --- Stage 3: Radiology ---
print(f"\n{'-'*50}")
print("STAGE 3: RADIOLOGY")
print(f"{'-'*50}")
print(f"Timestamp: {s['radiology'].get('timestamp', 'N/A')}")
print(f"Images:    {len(s['radiology'].get('images', []))}")
print(f"Reports:   {len(s['radiology'].get('reports', []))}")
for i, r in enumerate(s['radiology'].get('reports', []), 1):
    print(f"  Report {i}: {str(r.get('findings', 'N/A'))[:250]}...")

# --- Stage 4: Discharge ---
print(f"\n{'-'*50}")
print("STAGE 4: DISCHARGE")
print(f"{'-'*50}")
print(f"Timestamp:   {s['discharge'].get('timestamp', 'N/A')}")
print(f"ICD-10 Codes: {len(s['discharge'].get('icd10_codes', []))}")
for c in s['discharge'].get('icd10_codes', []):
    print(f"  {c['code']}: {c['description']}")
print(f"Medications at Discharge: {s['discharge'].get('medications_at_discharge', [])}")
print(f"Follow-up: {s['discharge'].get('follow_up', 'N/A')}")
ds = s['discharge'].get('summary', 'N/A')
print(f"\nDischarge Summary ({len(str(ds))} chars):")
print(f"  {str(ds)[:500]}{'...' if len(str(ds)) > 500 else ''}")

print(f"\n{'=' * 70}")
print("ENCOUNTER COMPLETE")
print(f"{'=' * 70}")